# 02. Data Preparation & Feature Engineering

Notebook ini menyiapkan dataset bersih untuk pemodelan Machine Learning:
- Membuang baris anomali dan kolom artefak
- Standarisasi 56 variasi ejaan wilayah (`ship-state`) ke bentuk kanonik
- Menghasilkan kamus pemetaan `ml/data/processed/state_mapping.json`
- Rekayasa fitur pre-dispatch: `unit_price`, fitur temporal, fitur biner operasional
- Menghasilkan target `is_cancelled`
- Mengekspor `orders_ml.csv` (level item/pesanan) dan `state_features.csv` (agregasi wilayah)


In [1]:
import pandas as pd
import numpy as np
import json
import os

RAW_DATA_PATH = "../data/raw/Amazon_Sale_Report_Cleaned.csv"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Raw dataset shape: {df.shape}")


Raw dataset shape: (42357, 29)


## 1. Pembersihan Baris Anomali & Kolom Artefak
Menghapus baris dummy anomali di akhir file (`Order ID: 405-2558167-3`), kolom artefak kosong `Unnamed: 22`, dan merapikan whitespace pada nama kolom.

In [2]:
# Hapus baris dummy anomali
initial_len = len(df)
df = df[df["Order ID"] != "405-2558167-3"].copy()
print(f"Dihapus {initial_len - len(df)} baris dummy anomali.")

# Hapus kolom kosong penuh
if "Unnamed: 22" in df.columns:
    df.drop(columns=["Unnamed: 22"], inplace=True)
    print("Kolom Unnamed: 22 berhasil dibuang.")

# Rapikan spasi nama kolom
df.columns = [c.strip() for c in df.columns]
print(f"Jumlah kolom saat ini: {len(df.columns)}")


Dihapus 1 baris dummy anomali.
Kolom Unnamed: 22 berhasil dibuang.
Jumlah kolom saat ini: 28


## 2. Standarisasi Wilayah (ship-state) & state_mapping.json
Memetakan seluruh 56 variasi penulisan nama wilayah (huruf kecil, singkatan, typo, gabungan teks) ke 36 nama negara bagian/teritori kanonik India.

In [3]:
state_mapping = {
    "ANDAMAN & NICOBAR ": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR": "Andaman and Nicobar Islands",
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "Bihar": "Bihar",
    "CHANDIGARH": "Chandigarh",
    "CHHATTISGARH": "Chhattisgarh",
    "Chandigarh": "Chandigarh",
    "DADRA AND NAGAR": "Dadra and Nagar Haveli",
    "DELHI": "Delhi",
    "Delhi": "Delhi",
    "New Delhi": "Delhi",
    "delhi": "Delhi",
    "GOA": "Goa",
    "Goa": "Goa",
    "goa": "Goa",
    "Gujarat": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "LADAKH": "Ladakh",
    "LAKSHADWEEP": "Lakshadweep",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "Manipur": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "NL": "Nagaland",
    "ODISHA": "Odisha",
    "Odisha": "Odisha",
    "orissa": "Odisha",
    "PB": "Punjab",
    "PUDUCHERRY": "Puducherry",
    "PUNJAB": "Punjab",
    "Puducherry": "Puducherry",
    "Punjab": "Punjab",
    "punjab": "Punjab",
    "Punjab/Mohali/Zirakpur": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "RJ": "Rajasthan",
    "Rajasthan": "Rajasthan",
    "Rajshthan": "Rajasthan",
    "rajasthan": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal"
}

# Simpan kamus pemetaan ke JSON
mapping_file = os.path.join(PROCESSED_DIR, "state_mapping.json")
with open(mapping_file, "w", encoding="utf-8") as f:
    json.dump(state_mapping, f, indent=2)
print(f"state_mapping.json berhasil disimpan ke {mapping_file}")

df["ship_state_clean"] = df["ship-state"].map(state_mapping).fillna("Unknown")
valid_count = df[df["ship_state_clean"] != "Unknown"]["ship_state_clean"].nunique()
print(f"Jumlah unique state kanonik: {valid_count}")


state_mapping.json berhasil disimpan ke ../data/processed\state_mapping.json


Jumlah unique state kanonik: 36


## 3. Imputasi Kuantitas & Rekayasa Unit Price
- Imputasi `Qty = 0` (yang terjadi saat pembatalan) menjadi kuantitas riil minimum checkout (1).
- Rekayasa `unit_price`: untuk pesanan yang dibatalkan (`Amount = 0`), estimasi harga satuan per unit menggunakan nilai median produk pada `SKU` terkait, atau fallback ke `Category`, lalu fallback ke median global.

In [4]:
# Imputasi Qty: nilai 0 diubah menjadi 1
df["qty_imputed"] = df["Qty"].apply(lambda x: 1 if x == 0 else int(x))

# Hitung raw unit price untuk transaksi yang memiliki Amount > 0 dan Qty > 0
df["raw_unit_price"] = np.where((df["Amount"] > 0) & (df["Qty"] > 0), df["Amount"] / df["Qty"], np.nan)

sku_median = df.groupby("SKU")["raw_unit_price"].median()
cat_median = df.groupby("Category")["raw_unit_price"].median()
overall_median = df["raw_unit_price"].median()

def compute_unit_price(row):
    if not np.isnan(row["raw_unit_price"]):
        return row["raw_unit_price"]
    sku = row.get("SKU")
    if sku in sku_median and not np.isnan(sku_median[sku]):
        return sku_median[sku]
    cat = row.get("Category")
    if cat in cat_median and not np.isnan(cat_median[cat]):
        return cat_median[cat]
    return overall_median

df["unit_price"] = df.apply(compute_unit_price, axis=1).round(2)
print("Ringkasan statistik unit_price:")
print(df["unit_price"].describe())
print(f"Missing unit_price: {df['unit_price'].isnull().sum()}")


Ringkasan statistik unit_price:
count    42356.000000
mean       635.846374
std        256.233819
min        199.000000
25%        435.000000
50%        575.000000
75%        788.000000
max       2299.000000
Name: unit_price, dtype: float64
Missing unit_price: 0


## 4. Rekayasa Fitur Pre-Dispatch (Biner & Temporal)
Mengekstrak prediktor yang valid saat checkout tanpa kebocoran data (*data leakage*):
- `order_date`, `day_of_week`, `day_name`, `is_weekend`, `day_of_month`
- `is_b2b` (dari kolom B2B)
- `is_fba` (Fulfilment == "Amazon")
- `has_promotion` (promotion-ids tidak kosong)
- `is_expedited` (ship-service-level == "Expedited")
- `is_cancelled` (Target: Status == "Cancelled")

In [5]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["day_of_week"] = df["order_date"].dt.dayofweek
df["day_name"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["day_of_month"] = df["order_date"].dt.day

df["is_b2b"] = df["B2B"].fillna(False).astype(bool).astype(int)
df["is_fba"] = (df["Fulfilment"] == "Amazon").astype(int)
df["has_promotion"] = df["promotion-ids"].notnull().astype(int)
df["is_expedited"] = (df["ship-service-level"] == "Expedited").astype(int)
df["is_cancelled"] = (df["Status"] == "Cancelled").astype(int)

print("Distribusi target is_cancelled:")
print(df["is_cancelled"].value_counts(normalize=True))


Distribusi target is_cancelled:
is_cancelled
0    0.855227
1    0.144773
Name: proportion, dtype: float64


## 5. Simpan Dataset Transaksi: orders_ml.csv
Menyimpan dataset transaksi siap model dengan fitur-fitur bersih tanpa nilai null.

In [6]:
ml_cols = [
    "Order ID",
    "order_date",
    "Category",
    "Size",
    "Style",
    "SKU",
    "Fulfilment",
    "is_fba",
    "ship-service-level",
    "is_expedited",
    "ship_state_clean",
    "Qty",
    "qty_imputed",
    "Amount",
    "unit_price",
    "is_b2b",
    "has_promotion",
    "day_of_week",
    "day_name",
    "is_weekend",
    "day_of_month",
    "is_cancelled"
]

orders_ml = df[ml_cols].copy()
orders_ml_file = os.path.join(PROCESSED_DIR, "orders_ml.csv")
orders_ml.to_csv(orders_ml_file, index=False)
print(f"orders_ml.csv berhasil disimpan ke {orders_ml_file} dengan shape: {orders_ml.shape}")

# Validasi null pada fitur model
feature_cols = ["Category", "Size", "is_fba", "is_expedited", "ship_state_clean", "qty_imputed", "unit_price", "is_b2b", "has_promotion", "is_weekend", "is_cancelled"]
null_check = orders_ml[feature_cols].isnull().sum()
print("Pemeriksaan nilai kosong fitur utama:")
print(null_check)
assert null_check.sum() == 0, "Ditemukan nilai NaN pada kolom fitur utama!"


orders_ml.csv berhasil disimpan ke ../data/processed\orders_ml.csv dengan shape: (42356, 22)
Pemeriksaan nilai kosong fitur utama:
Category            0
Size                0
is_fba              0
is_expedited        0
ship_state_clean    0
qty_imputed         0
unit_price          0
is_b2b              0
has_promotion       0
is_weekend          0
is_cancelled        0
dtype: int64


## 6. Agregasi Metrik Per Wilayah: state_features.csv
Menghitung ringkasan karakteristik per wilayah (negara bagian) untuk segmentasi K-Means pada Fase 3.

In [7]:
# Exclude Unknown state untuk klaster spasial
valid_states = df[df["ship_state_clean"] != "Unknown"].copy()

state_agg = valid_states.groupby("ship_state_clean").agg(
    total_orders=("Order ID", "count"),
    total_revenue=("Amount", "sum"),
    cancellation_rate=("is_cancelled", "mean"),
    avg_unit_price=("unit_price", "mean"),
    fba_ratio=("is_fba", "mean"),
    b2b_ratio=("is_b2b", "mean"),
    expedited_ratio=("is_expedited", "mean"),
    promotion_ratio=("has_promotion", "mean"),
    avg_qty=("qty_imputed", "mean")
).reset_index()

state_agg.rename(columns={"ship_state_clean": "state"}, inplace=True)
state_agg["cancellation_rate"] = state_agg["cancellation_rate"].round(4)
state_agg["avg_unit_price"] = state_agg["avg_unit_price"].round(2)
state_agg["fba_ratio"] = state_agg["fba_ratio"].round(4)
state_agg["b2b_ratio"] = state_agg["b2b_ratio"].round(4)
state_agg["expedited_ratio"] = state_agg["expedited_ratio"].round(4)
state_agg["promotion_ratio"] = state_agg["promotion_ratio"].round(4)
state_agg["avg_qty"] = state_agg["avg_qty"].round(3)

state_features_file = os.path.join(PROCESSED_DIR, "state_features.csv")
state_agg.to_csv(state_features_file, index=False)
print(f"state_features.csv berhasil disimpan ke {state_features_file} dengan {len(state_agg)} negara bagian/UT.")
assert len(state_agg) >= 25, f"Jumlah state {len(state_agg)} kurang dari ambang batas minimal 25!"
print("Preview state_features:")
print(state_agg.head(10))


state_features.csv berhasil disimpan ke ../data/processed\state_features.csv dengan 36 negara bagian/UT.
Preview state_features:
                         state  total_orders  total_revenue  \
0  Andaman and Nicobar Islands            79       46794.78   
1               Andhra Pradesh          1679      954802.94   
2            Arunachal Pradesh            43       26543.38   
3                        Assam           641      368137.69   
4                        Bihar           756      492083.30   
5                   Chandigarh           102       67719.33   
6                 Chhattisgarh           316      194166.56   
7       Dadra and Nagar Haveli            25       14422.90   
8                        Delhi          2347     1434950.06   
9                          Goa           408      233106.05   

   cancellation_rate  avg_unit_price  fba_ratio  b2b_ratio  expedited_ratio  \
0             0.2405          657.77     0.6582     0.0000           0.6582   
1             0.163